## Week 6: Agentic RAG

In [1]:
import os
import re
import sys
import time
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, "../src")

import pandas as pd
from dotenv import load_dotenv
from typing import List, Any
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder
from langgraph.graph import StateGraph, END
from utils.doc_preprocessing import extract_sections, get_breadcrumb

load_dotenv("../.env")

True

## 1. Baseline (Hybrid+Re-ranking)

In [4]:
PDF_PATH = "../data/registration_of_real_estatee_manual.pdf"
sections, all_headings = extract_sections(PDF_PATH)

section_docs = [
    Document(
        page_content=re.sub(r'\n+', ' ', s["content"]).strip(),
        metadata={
            "section_num": s["num"],
            "section_title": s["title"],
            "breadcrumb": get_breadcrumb(s["num"], all_headings),
            "start_page": s["start_page"],
        },
    )
    for s in sections
    if s["content"].strip()
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = [c for c in text_splitter.split_documents(section_docs) if c.page_content.strip()]
print(f"총 청크 수: {len(chunks)}")

총 청크 수: 250


In [5]:
DENSE_DB_PATH = "../chroma_db/real_estate_RAG"
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# 실제 문서가 저장된 컬렉션 이름은 "real_estate_RAG" (대문자)
COLLECTION_NAME = "real_estate_RAG"

if os.path.exists(DENSE_DB_PATH) and os.listdir(DENSE_DB_PATH):
    db = Chroma(
        persist_directory=DENSE_DB_PATH,
        embedding_function=embeddings,
        collection_name=COLLECTION_NAME,
    )
    print(f"기존 ChromaDB 로드: {DENSE_DB_PATH}")
else:
    db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DENSE_DB_PATH,
        collection_name=COLLECTION_NAME,
    )
    print("ChromaDB 신규 생성 완료")

print(f"컬렉션 내 문서 수: {db._collection.count()}")

dense_retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 20},
)
print("Dense retriever 준비 완료")

기존 ChromaDB 로드: ../chroma_db/real_estate_RAG
컬렉션 내 문서 수: 250
Dense retriever 준비 완료


In [6]:
def korean_tokenizer(text: str):
    """BM25용 한국어 토크나이저: 특수문자 제거 + 공백 분리 + 1글자 제거"""
    cleaned = re.sub("[^가-힣a-zA-Z0-9]", " ", text)
    return [t for t in cleaned.split() if len(t) > 1]

bm25_retriever = BM25Retriever.from_documents(
    chunks,
    k=10,
    preprocess_func=korean_tokenizer,
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
    c=60,
)

In [7]:
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
cross_encoder = CrossEncoder(RERANKER_MODEL)

# Reranker용 k=20 후보 풀 (week5와 동일하게)
bm25_retriever_k20 = BM25Retriever.from_documents(
    chunks,
    k=10,
    preprocess_func=korean_tokenizer,
)
dense_retriever_k20 = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 20},
)
hybrid_retriever_k20 = EnsembleRetriever(
    retrievers=[bm25_retriever_k20, dense_retriever_k20],
    weights=[0.5, 0.5],
    c=60,
)

def hybrid_rerank_retriever(query: str, top_k: int = 5) -> List[Document]:
    """Hybrid top-20 1차 검색 → Cross-Encoder top-5 재정렬"""
    candidates = hybrid_retriever_k20.invoke(query)
    if not candidates:
        return []
    pairs = [(query, doc.page_content) for doc in candidates]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 529.90it/s]


## 2. LangGraph


- 모든 Node에 공유되는 메모리: **State**

In [8]:
class GraphState(TypedDict):
    question: str            # 원래 질문
    rewritten_question: str  # query rewrite 후 질문
    documents: List[Any]     # 검색된 문서 목록
    answer: str              # 생성된 답변
    grade_result: str        # 'yes' 또는 'no'
    retry_count: int         # 쿼리 재작성 Count 
    route_history: list      # 라우팅 경로 추적
    latency: float           # 총 처리 시간

workflow = StateGraph(GraphState)
MAX_RETRIES = 2

- 하나의 처리 단계: **Node**          
    - input: State -> output: State (update)

In [9]:
# Node 1

def retrieve(state: GraphState) -> dict:
    q = state.get("rewritten_question") or state["question"]
    docs = hybrid_rerank_retriever(q, top_k=5)
    # history = list(state.get("route_history") or [])
    # history.append(f"retrieve(q='{q[:30]}...')")
    print(f"[retrieve] 쿼리: {q} | 문서: {len(docs)}개")
    return {
        "documents": docs,
        "grade_result": "",
        "answer": "",
        # "route_history": history,
    }

In [10]:
# Node 2

from prompt.prompt import GRADE_PROMPT

class GradeResult(BaseModel):
    """pydantic으로 구조화된 답변 내뱉도록"""
    relevance: str = Field(description="'yes'/'no'")
    reason: str = Field(description="판단 이유)")
    
    
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
grade_llm = llm.with_structured_output(GradeResult)


def grade_documents(state: GraphState) -> dict:

    q = state.get("rewritten_question") or state["question"]
    docs = state["documents"]

    if not docs:
        return {"grade_result": "no"}

    doc_previews = "\n\n".join([
        f"[문서 {i+1}] 출처: {doc.metadata.get('breadcrumb', 'N/A')}\n{doc.page_content[:250]}"
        for i, doc in enumerate(docs[:5]) # 최대 5개 문서만 프롬프트에 포함
    ])

    grade_input = GRADE_PROMPT.format_messages(question=q, doc_previews=doc_previews)
    result = grade_llm.invoke(grade_input)

    # history = list(state.get("route_history") or [])
    # history.append(f"grade={result.relevance} ({result.reason[:40]})")
    # print(f"[grade] {result.relevance} | {result.reason}")

    return {"grade_result": result.relevance} #, "route_history": history}

In [11]:
# Node 3

from prompt.prompt import REWRITE_PROMPT

def rewrite_query(state: GraphState) -> dict:
    current_q = state.get("rewritten_question") or state["question"]
    retry_count = state.get("retry_count") or 0

    rewrite_input = REWRITE_PROMPT.format_messages(question=current_q)
    response = llm.invoke(rewrite_input)
    rewritten = response.content.strip()

    new_retry = retry_count + 1
    # history = list(state.get("route_history") or [])
    # history.append(f"rewrite({new_retry}/{MAX_RETRIES}): '{rewritten[:40]}'")

    print(f"[rewrite] {new_retry}/{MAX_RETRIES}회")
    print(f"  원래: {current_q}")
    print(f"  재작성: {rewritten}")
    return {
        "rewritten_question": rewritten,
        "retry_count": new_retry,
        # "route_history": history,
    }

In [12]:
# Node 4

from prompt.prompt import GENERATE_PROMPT

def generate(state: GraphState) -> dict:
    q = state.get("rewritten_question") or state["question"]
    docs = state.get("documents") or []
    grade_result = state.get("grade_result", "no")
    retry_count = state.get("retry_count") or 0

    if grade_result != "yes" or not docs:
        fallback = (
            f"(재검색 {retry_count}회 시도, 모두 관련 문서 부족)\n"
            f"추가 문서가 제공되면 더 정확히 답변할 수 있습니다."
        )
        print(f"[generate] 답변 불가")
        return {"answer": fallback}

    context = "\n\n".join([doc.page_content for doc in docs])
    messages = GENERATE_PROMPT.format_messages(context=context, question=q)
    response = llm.invoke(messages)
    print(f"[generate] 답변 생성 완료 ({len(response.content)}자)")
    return {"answer": response.content}

In [13]:
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("rewrite_query", rewrite_query)
workflow.add_node("generate", generate)

- 다음 흐름을 결정: **Edge**
    - 일반 Edge
    - Conditional Edge

In [14]:
def route_after_grade(state: GraphState) -> str:
    """
    grade 결과 + retry_count에 따라 다음 노드 결정
    """
    grade = state.get("grade_result", "insufficient")
    retry = state.get("retry_count") or 0

    if grade == "yes":
        return "generate"
    elif retry >= MAX_RETRIES:
        return "generate"   # generate 내부에서 답변 불가 처리
    else:
        return "rewrite_query"

workflow.set_entry_point("retrieve") # 시작 노드 
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    route_after_grade, # Router: 함수를 호출해서 다음 노드를 결정
    {"generate": "generate", "rewrite_query": "rewrite_query"}, # Path map: 함수 반환 String을 노드명에 매핑
)
workflow.add_edge("rewrite_query", "retrieve")
workflow.add_edge("generate", END)

app = workflow.compile()

In [15]:
# Workflow 시각화
mermaid_str = app.get_graph().draw_mermaid()
print(mermaid_str)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade_documents(grade_documents)
	rewrite_query(rewrite_query)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	grade_documents -.-> generate;
	grade_documents -.-> rewrite_query;
	retrieve --> grade_documents;
	rewrite_query --> retrieve;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [16]:
def run_agentic(query: str, verbose: bool = True) -> dict:
    """Agentic RAG 실행 래퍼"""
    if verbose:
        print(f"\n{'='*65}")
        print(f"[질문] {query}")
        print('='*65)

    start = time.time()
    initial_state: GraphState = {
        "question": query,
        "rewritten_question": "",
        "documents": [],
        "answer": "",
        "grade_result": "",
        "retry_count": 0,
        "latency": 0.0,
    }
    final_state = app.invoke(initial_state)
    latency = time.time() - start

    final_docs = final_state.get("documents") or []

    if verbose:
        print(f"\n[최종 답변]\n{final_state['answer']}")
        print(f"\n[Latency] {latency:.2f}s | 재시도 {final_state.get('retry_count', 0)}회")

    return {
        "answer": final_state["answer"],
        "contexts": [doc.page_content for doc in final_docs],
        "breadcrumbs": [doc.metadata.get("breadcrumb", "N/A") for doc in final_docs],
        "grade_result": final_state.get("grade_result", ""),
        "retry_count": final_state.get("retry_count", 0),
        "rewritten_question": final_state.get("rewritten_question", ""),
        "latency": latency,
    }

## 3. 결과 비교: RAG vs Agentic RAG

In [17]:
os.environ["RAGAS_MAX_CONCURRENCY"] = "1"

from langchain_google_genai import ChatGoogleGenerativeAI
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithoutReference,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0,
    model_kwargs={"seed": 42},
)
ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
]


TEST_QUESTIONS = [
    "부모님이 살아계실 때 집을 미리 자식한테 넘겨주고 싶대요. 상속이랑 다른 건가요? 등기는 어떻게 해요?",
    "새로 지은 건물 처음 등기할 때 뭐가 필요해요?",
    "미성년자가 상속으로 부동산을 취득했을 때 등기 신청은 누가 하나요?",
    "근저당권 말소등기 절차와 준비서류를 알려주세요",
    "가등기 해놨는데 잔금 다 내고 나서 본등기로 바꾸는 방법이 뭐예요?"
]

print(f"RAGAS 설정 완료 | 테스트 질문 {len(TEST_QUESTIONS)}개")
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"  Q{i}: {q[:60]}")

RAGAS 설정 완료 | 테스트 질문 5개
  Q1: 부모님이 살아계실 때 집을 미리 자식한테 넘겨주고 싶대요. 상속이랑 다른 건가요? 등기는 어떻게 해요?
  Q2: 새로 지은 건물 처음 등기할 때 뭐가 필요해요?
  Q3: 미성년자가 상속으로 부동산을 취득했을 때 등기 신청은 누가 하나요?
  Q4: 근저당권 말소등기 절차와 준비서류를 알려주세요
  Q5: 가등기 해놨는데 잔금 다 내고 나서 본등기로 바꾸는 방법이 뭐예요?


In [19]:
def run_baseline(query: str) -> dict:
    """Baseline: Hybrid + Reranker, LangGraph 없이 직접 실행"""
    start = time.time()
    docs = hybrid_rerank_retriever(query, top_k=5)
    context = "\n\n".join([doc.page_content for doc in docs])
    messages = GENERATE_PROMPT.format_messages(context=context, question=query)
    response = llm.invoke(messages)
    return {
        "answer": response.content,
        "contexts": [doc.page_content for doc in docs],
        "breadcrumbs": [doc.metadata.get("breadcrumb", "N/A") for doc in docs],
        "latency": time.time() - start,
    }

In [20]:
print("[Baseline: Hybrid + Reranker] 평가 시작...")
baseline_results = []
for i, q in enumerate(TEST_QUESTIONS):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {q}")
    res = run_baseline(q)
    baseline_results.append(res)
    print(f"  컨텍스트 수: {len(res['contexts'])}개")
    for j, bc in enumerate(res['breadcrumbs'], 1):
        print(f"    [{j}] {bc}")
    print(f"  답변: {res['answer'][:200]}")
    print(f"  Latency: {res['latency']:.2f}s")

baseline_samples = [
    SingleTurnSample(
        user_input=q,
        response=r["answer"],
        retrieved_contexts=r["contexts"],
    )
    for q, r in zip(TEST_QUESTIONS, baseline_results)
]
baseline_eval = evaluate(
    dataset=EvaluationDataset(samples=baseline_samples),
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
baseline_df = baseline_eval.to_pandas()
print("\nBaseline 평균 점수")
print(baseline_df[["faithfulness", "answer_relevancy", "llm_context_precision_without_reference"]].mean())

[Baseline: Hybrid + Reranker] 평가 시작...

Q1: 부모님이 살아계실 때 집을 미리 자식한테 넘겨주고 싶대요. 상속이랑 다른 건가요? 등기는 어떻게 해요?
  컨텍스트 수: 5개
    [1] 소유권이전등기 > 상속에의한소유권이전등기 > 개념및신청인
    [2] 소유권이전등기 > 상속에의한소유권이전등기 > 개념및신청인
    [3] 소유권이전등기 > 증여에의한소유권이전등기 > 개념및신청인
    [4] 가등기 > 가등기에기한소유권이전본등기 > 개념및신청인
    [5] 소유권이전등기 > 상속에의한소유권이전등기 > 제출서류
  답변: 부모님이 살아계실 때 집을 자식에게 넘겨주는 것은 '증여'에 해당합니다. 상속은 사망으로 인해 재산이 이전되는 경우를 말하며, 증여는 생존 중에 재산을 무상으로 이전하는 것입니다. 

증여에 의한 소유권 이전등기를 하려면, 다음과 같은 절차를 따라야 합니다:

1. **증여계약서 작성**: 부모님과 자식 간에 증여계약서를 작성해야 합니다. 이 계약서에는 증
  Latency: 271.49s

Q2: 새로 지은 건물 처음 등기할 때 뭐가 필요해요?
  컨텍스트 수: 5개
    [1] 건물멸실등기 > 건물멸실등기 > 개념및신청인
    [2] 건물멸실등기 > 건물멸실등기 > 신청절차
    [3] 소유권보존등기 > 건물소유권보존등기 > 개념및신청인
    [4] 건물멸실등기 > 건물멸실등기 > 제출서류
    [5] 소유권보존등기 > 구분건물소유권보존등기 > 제출서류
  답변: 새로 지은 건물을 처음 등기할 때 필요한 서류는 다음과 같습니다:

1. **신청인의 소유권을 증명하는 서면**: 건축물대장 등본이 필요합니다. 이는 해당 건물이 건축물대장에 등록되어 있어야 함을 증명합니다.

2. **신청인의 주소를 증명하는 서면**: 주민등록등본이 필요합니다. 법인의 경우에는 법인등기사항증명서를 제출해야 합니다.

3. **소유권 보존
  Latency: 162.40s

Q3: 미성년자가 상속으로 부동산을 취득

Evaluating:  13%|█▎        | 2/15 [00:04<00:26,  2.05s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 15/15 [00:56<00:00,  3.80s/it]


Baseline 평균 점수
faithfulness                               0.596992
answer_relevancy                           0.821656
llm_context_precision_without_reference    0.580000
dtype: float64


In [21]:
print("[Agentic RAG] 평가 시작...")
agentic_results = []
for i, q in enumerate(TEST_QUESTIONS):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {q}")
    res = run_agentic(q, verbose=False)
    agentic_results.append(res)
    print(f"  grade={res['grade_result']}, retry={res['retry_count']}")
    if res['rewritten_question']:
        print(f"  rewritten: {res['rewritten_question'][:60]}")
    print(f"  컨텍스트 수: {len(res['contexts'])}개")
    for j, bc in enumerate(res['breadcrumbs'], 1):
        print(f"    [{j}] {bc}")
    print(f"  답변: {res['answer'][:200]}")
    print(f"  Latency: {res['latency']:.2f}s")

agentic_samples = [
    SingleTurnSample(
        user_input=q,
        response=r["answer"],
        retrieved_contexts=r["contexts"] if r["contexts"] else ["검색 결과 없음"],
    )
    for q, r in zip(TEST_QUESTIONS, agentic_results)
]
agentic_eval = evaluate(
    dataset=EvaluationDataset(samples=agentic_samples),
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
agentic_df = agentic_eval.to_pandas()
print("\nAgentic RAG 평균 점수")
print(agentic_df[["faithfulness", "answer_relevancy", "llm_context_precision_without_reference"]].mean())

[Agentic RAG] 평가 시작...

Q1: 부모님이 살아계실 때 집을 미리 자식한테 넘겨주고 싶대요. 상속이랑 다른 건가요? 등기는 어떻게 해요?
[retrieve] 쿼리: 부모님이 살아계실 때 집을 미리 자식한테 넘겨주고 싶대요. 상속이랑 다른 건가요? 등기는 어떻게 해요? | 문서: 5개
[rewrite] 1/2회
  원래: 부모님이 살아계실 때 집을 미리 자식한테 넘겨주고 싶대요. 상속이랑 다른 건가요? 등기는 어떻게 해요?
  재작성: 부모가 생전에 자녀에게 부동산을 증여하고자 할 때, 상속과의 차이점은 무엇이며, 해당 부동산의 증여등기 절차는 어떻게 진행되나요?
[retrieve] 쿼리: 부모가 생전에 자녀에게 부동산을 증여하고자 할 때, 상속과의 차이점은 무엇이며, 해당 부동산의 증여등기 절차는 어떻게 진행되나요? | 문서: 5개
[generate] 답변 생성 완료 (797자)
  grade=yes, retry=1
  rewritten: 부모가 생전에 자녀에게 부동산을 증여하고자 할 때, 상속과의 차이점은 무엇이며, 해당 부동산의 증여등기 절차
  컨텍스트 수: 5개
    [1] 소유권이전등기 > 상속에의한소유권이전등기 > 제출서류
    [2] 소유권이전등기 > 상속에의한소유권이전등기 > 개념및신청인
    [3] 소유권이전등기 > 증여에의한소유권이전등기 > 제출서류
    [4] 가등기 > 가등기에기한소유권이전본등기 > 개념및신청인
    [5] 소유권이전등기 > 증여에의한소유권이전등기 > 개념및신청인
  답변: 부모가 생전에 자녀에게 부동산을 증여하고자 할 때, 상속과의 차이점은 다음과 같습니다:

1. **상속**: 상속은 피상속인이 사망한 후 그 재산이 법정 상속인에게 이전되는 과정입니다. 상속은 법률에 따라 정해진 상속분에 따라 이루어지며, 상속세가 부과됩니다. 상속인은 피상속인의 사망 시점에 존재해야 하며, 상속 절차는 상속인 전원이 참여해야 합니다.

2
  Latency: 358.21s

Q2: 새로 지은 건물 처음 등기할 

Evaluating:   7%|▋         | 1/15 [00:03<00:52,  3.78s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  13%|█▎        | 2/15 [00:11<01:22,  6.31s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 15/15 [01:01<00:00,  4.08s/it]


Agentic RAG 평균 점수
faithfulness                               0.703324
answer_relevancy                           0.808822
llm_context_precision_without_reference    0.406667
dtype: float64


In [24]:
# ── 최종 비교 테이블 ──
b_scores = baseline_df[["faithfulness", "answer_relevancy", "llm_context_precision_without_reference"]].mean()
a_scores  = agentic_df[["faithfulness", "answer_relevancy", "llm_context_precision_without_reference"]].mean()

b_lat = sum(r["latency"] for r in baseline_results) / len(baseline_results)
a_lat = sum(r["latency"] for r in agentic_results)  / len(agentic_results)

comparison = pd.DataFrame([
    {
        "구성": "Baseline Hybrid+Reranker)",
        "Faithfulness": round(b_scores["faithfulness"], 3),
        "Answer Relevancy": round(b_scores["answer_relevancy"], 3),
        "Context Precision": round(b_scores["llm_context_precision_without_reference"], 3),
        "평균 Latency(s)": round(b_lat, 2),
    },
    {
        "구성": "Agentic RAG (LangGraph)",
        "Faithfulness": round(a_scores["faithfulness"], 3),
        "Answer Relevancy": round(a_scores["answer_relevancy"], 3),
        "Context Precision": round(a_scores["llm_context_precision_without_reference"], 3),
        "평균 Latency(s)": round(a_lat, 2),
    },
])

print("\n" + "="*72)
print("Baseline vs Agentic RAG 최종 비교")
print("="*72)
print(comparison.to_string(index=False))


Baseline vs Agentic RAG 최종 비교
                       구성  Faithfulness  Answer Relevancy  Context Precision  평균 Latency(s)
Baseline Hybrid+Reranker)         0.597             0.822              0.580         182.30
  Agentic RAG (LangGraph)         0.703             0.809              0.407         179.08


In [26]:
# Retry 통계
retry_counts = [r["retry_count"] for r in agentic_results]
print("\nAgentic RAG Retry 통계")
for c in range(MAX_RETRIES + 1):
    print(f"  재시도 {c}회: {retry_counts.count(c)}건")

print("\n각 질문별 상세 결과")
for i, (q, r) in enumerate(zip(TEST_QUESTIONS, agentic_results)):
    print(f"  Q{i+1}: retry={r['retry_count']}, grade={r['grade_result']}, latency={r['latency']:.2f}s")
    if r["rewritten_question"]:
        print(f"        rewritten: {r['rewritten_question'][:60]}")


Agentic RAG Retry 통계
  재시도 0회: 4건
  재시도 1회: 1건
  재시도 2회: 0건

각 질문별 상세 결과
  Q1: retry=1, grade=yes, latency=358.21s
        rewritten: 부모가 생전에 자녀에게 부동산을 증여하고자 할 때, 상속과의 차이점은 무엇이며, 해당 부동산의 증여등기 절차
  Q2: retry=0, grade=yes, latency=131.43s
  Q3: retry=0, grade=yes, latency=127.71s
  Q4: retry=0, grade=yes, latency=147.25s
  Q5: retry=0, grade=yes, latency=130.82s


## 4. 실패케이스 



In [ ]:
# 질문 유형: 비교, 복합 조건, 다단계 추론
TEST_QUESTIONS = [
    # "전세권 설정등기와 임차권 등기명령의 차이점은 무엇이며, 각각 어떤 상황에서 선택하는 것이 유리한가요?",
    # "공동 상속인이 여러 명인 경우 부동산 상속등기 신청은 어떻게 진행되며, 상속인 중 미성년자가 포함된 경우 절차가 어떻게 달라지나요?",
    # "이전등기 신청 시 인감증명서 제출이 필요한 경우와 면제되는 경우는 각각 어떤 상황인가요?",
    # "부동산 매매 계약 후 소유권 이전등기를 완료하기 위해 매도인과 매수인이 각각 준비해야 할 서류와 역할은 무엇인가요?",
    # "근저당권이 설정된 부동산을 상속받았을 때 상속등기와 근저당권 처리를 어떤 순서로 진행해야 하나요?", # 답변불가 (context 부족)
]